# 02 - Supervised Fine-Tuning (SFT) & LLM-JEPA Multi-Task Training (FINER-SQL-3B)
## Text-to-SQL SLM Research Architecture (Phase 2: SFT & JEPA Alignment)

This notebook implements the dual-objective training pipeline for **FINER-SQL-3B**:
1. **GPU Environment & Hardware Detection**: Automatically profiles T4 / L4 / A100 GPUs.
2. **Google Drive Checkpoint Sync**: Automated mounting to `/content/drive` and session sync to `/content/drive/MyDrive/Text2SQL_Colab/checkpoints/sft_jepa_3b/`.
3. **Session Recovery**: Automated crash recovery via `--resume_from_checkpoint latest`.
4. **LoRA PEFT Setup**: Multi-target LoRA ($r=64$, $\alpha=16$) on 7 linear projection modules.
5. **Completion-Only Cross-Entropy Loss**: Masking prompt tokens with $-100$ before the `<think>` tag to prevent structural memorization.
6. **LLM-JEPA Alignment Module**: 2-block causal mask with `[PRED]` token and random loss dropout ($LD=0.5$).
7. **DriveCheckpointCallback**: Seamless Google Drive checkpoint persistence.


In [ ]:
# Step 1: Workspace Path Setup & GPU Hardware Detection
import os
import sys

WORKSPACE_DIR = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.exists("../src") else os.path.abspath(".")
if WORKSPACE_DIR not in sys.path:
    sys.path.insert(0, WORKSPACE_DIR)

import json
import time
import subprocess

try:
    import torch
    gpu_available = torch.cuda.is_available()
    device_name = torch.cuda.get_device_name(0) if gpu_available else "CPU (Fallback)"
    device_count = torch.cuda.device_count() if gpu_available else 0
except ImportError:
    torch = None
    gpu_available = False
    device_name = "CPU (Fallback)"
    device_count = 0

print("=== Google Colab GPU & Environment Verification ===")
print(f"CUDA Available:  {gpu_available}")
print(f"GPU Name:        {device_name} (Count: {device_count})")
if gpu_available and torch is not None:
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Total VRAM:      {vram_gb:.2f} GB")
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("Running in CPU testing mode.")

In [ ]:
# Step 2: Google Drive Automated Mounting & Checkpoint Path Setup
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = "/content/drive/MyDrive/Text2SQL_Colab/checkpoints/sft_jepa_3b"
    print("Google Drive successfully mounted at /content/drive")
except (ImportError, Exception):
    DRIVE_DIR = os.path.abspath("./colab_drive_mock/checkpoints/sft_jepa_3b")
    print(f"Local environment detected: using fallback drive path: {DRIVE_DIR}")

LOCAL_DIR = "/content/checkpoints/sft_jepa_3b" if os.path.exists("/content") else os.path.abspath("./checkpoints/sft_jepa_3b")

os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(LOCAL_DIR, exist_ok=True)

print(f"Drive Checkpoint Directory: {DRIVE_DIR}")
print(f"Local Checkpoint Directory: {LOCAL_DIR}")


In [ ]:
# Step 3: Session Recovery (--resume_from_checkpoint latest)
from src.training.callbacks import find_latest_checkpoint, get_resume_checkpoint

# CLI argument support: --resume_from_checkpoint latest
RESUME_FLAG = "latest"

resume_checkpoint_path = get_resume_checkpoint(
    resume_flag=RESUME_FLAG,
    local_dir=LOCAL_DIR,
    drive_dir=DRIVE_DIR
)

if resume_checkpoint_path:
    print(f"[RECOVERY] Discovered latest checkpoint: {resume_checkpoint_path}")
    print("Training will resume state, optimizer, and scheduler from this checkpoint.")
else:
    print("[NEW SESSION] No existing checkpoint found. Initializing fresh training run.")


In [ ]:
# Step 4: FINER-SQL-3B Model & LoRA PEFT Configuration
from src.training.train_sft_generator import build_lora_config

MODEL_ID_3B = "meta-llama/Llama-3.2-3B-Instruct"
LORA_R = 64
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

lora_config = build_lora_config(
    rank=LORA_R,
    alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES
)

print(f"Base Model:       {MODEL_ID_3B}")
print(f"LoRA Rank (r):    {lora_config.r}")
print(f"LoRA Alpha:       {lora_config.lora_alpha}")
print(f"Target Modules:   {lora_config.target_modules}")

In [ ]:
# Step 5: Completion-Only Data Collator (Masking Prompt Tokens with -100)
from src.training.train_sft_generator import CompletionOnlyDataCollator, mask_prompt_tokens, compute_completion_loss

DATA_PATH = "data/mini/mini_synsql.jsonl" if os.path.exists("data/mini/mini_synsql.jsonl") else "../data/mini/mini_synsql.jsonl"

with open(DATA_PATH, "r", encoding="utf-8") as f:
    training_samples = [json.loads(line) for line in f]

print(f"Loaded {len(training_samples)} multi-step reasoning samples from {DATA_PATH}")

# Demonstrate token masking: prompt tokens receive label -100
sample_prompt = training_samples[0]["formatted_prompt"]
prompt_end_pos = sample_prompt.find("<think>")
tokens = list(range(100))
masked_labels, _ = mask_prompt_tokens(tokens, prompt_end_token_idx=35)

print(f"Sample prompt length: {len(sample_prompt)} characters")
print(f"Tokens masked with -100: {sum(1 for x in masked_labels if x == -100)} tokens")
print(f"Tokens contributing to loss: {sum(1 for x in masked_labels if x != -100)} tokens")
print("Completion-Only masking verified: prompt tokens do not leak into loss gradient.")

In [ ]:
# Step 6: LLM-JEPA Alignment Module (2-Block Causal Mask & [PRED] Token)
from src.training.jepa_module import build_2block_causal_mask, LLM_JEPAModule, compute_jepa_loss

print("Verifying 2-Block Causal Mask structure...")
T_text = 24  # Text prompt tokens + [PRED] token
C_code = 36  # Target SQL / code tokens

mask = build_2block_causal_mask(seq_len_text=T_text, seq_len_code=C_code)
print(f"2-Block Causal Mask shape: {mask.shape}")

# Assert causality properties
assert mask[0, T_text + 5].item() < -1e8, "Block 1 must not attend to Block 2!"
assert mask[T_text + 5, 0].item() == 0.0, "Block 2 must attend to Block 1!"
print("[SUCCESS] 2-Block Causal Mask verified: strict non-leaking representation.")

jepa_module = LLM_JEPAModule(hidden_dim=256, predictor_dim=256, loss_dropout_prob=0.5)
print(f"Initialized LLM_JEPAModule (loss_dropout_prob={jepa_module.loss_dropout_prob})")


In [ ]:
# Step 7: DriveCheckpointCallback Configuration
from src.training.callbacks import DriveCheckpointCallback

drive_callback = DriveCheckpointCallback(
    drive_checkpoint_dir=DRIVE_DIR,
    max_drive_checkpoints=5,
    sync_every_n_steps=100
)

print(f"DriveCheckpointCallback configured:")
print(f"  Target Google Drive:      {drive_callback.drive_dir}")
print(f"  Max Checkpoints to keep:  {drive_callback.max_drive_checkpoints}")
print(f"  Sync frequency:           every {drive_callback.sync_every_n_steps} steps")

In [ ]:
# Step 8: Execute SFT + JEPA Training Loop
from src.training.train_sft_generator import run_sft_training

print("Starting SFT + JEPA Generator Training Pipeline...")
train_summary = run_sft_training(
    data_path=DATA_PATH,
    output_dir=LOCAL_DIR,
    drive_dir=DRIVE_DIR,
    epochs=3,
    batch_size=2,
    learning_rate=2e-4,
    dry_run=True  # Dry-run validates forward pass, loss calculation and checkpointing
)

print("\n=== Training Results Summary ===")
print(json.dumps(train_summary, indent=2))


## Checkpoint Status & Transition
- Model checkpoints synchronized to Google Drive: `/content/drive/MyDrive/Text2SQL_Colab/checkpoints/sft_jepa_3b/`.
- SFT weights are ready for Reinforcement Learning policy optimization.
- **Proceed to Notebook 03 (`notebooks/03_grpo_training_3b_and_7b.ipynb`)** for Dense GRPO training.
